# 09 – Pricing: Fourier vs Monte Carlo

Validates the Fourier pricing formula (Proposition 4.3 of the paper) against
Monte Carlo simulation for the calibrated coupled CARMA model.

### Payoffs tested
1. **Plain energy call** $(S_T - K_S)^+$
2. **Indicator quanto** $\mathbf{1}_{S_T > K_S}\,\mathbf{1}_{Y_T > K_Y}$
3. **Linear quanto call** $(S_T - K_S)^+\cdot(Y_T - K_Y)$ (signed, long)

### Method
- **MC**: exact matrix-exponential propagator with NIG increments (`simulate_coupled_carma`)
- **Fourier**: Carr-Madan FFT for call; 2D quadrature for indicator quanto (`fourier_price_indicator_quanto`)

Inputs:  `results/carma_temp_mle.json`, `results/carma_price_mle.json`,
         `results/levy_params_temperature.json`, `results/levy_params_logprice.json`,
         `results/coupling_params.json`  
Outputs: `results/pricing_comparison.csv`, `figures/pricing_mc_comparison.png`

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import RES_DIR, FIG_DIR, DT_YEARS, HOURS_PER_YEAR
from carma_utils import (
    build_companion, load_params,
    simulate_coupled_carma,
    carma_joint_cf, compute_forward_price,
    fourier_price_1d, fourier_price_indicator_quanto,
    _kernel_integrals,
)

# ── Load fitted parameters ────────────────────────────────────────────────────
params_temp  = load_params(RES_DIR / 'carma_temp_mle.json')
params_price = load_params(RES_DIR / 'carma_price_mle.json')

with open(RES_DIR / 'levy_params_temperature.json') as f:
    levy_temp  = json.load(f)
with open(RES_DIR / 'levy_params_logprice.json') as f:
    levy_price = json.load(f)
with open(RES_DIR / 'coupling_params.json') as f:
    coupling   = json.load(f)

nig_Y = levy_temp['nig']
nig_X = levy_price['nig']

# ── Build state-space matrices ────────────────────────────────────────────────
a_X = np.array(params_price['a_coeffs'])
b_X = np.array(params_price['b_coeffs'])
p_X = len(a_X)
A_X = build_companion(a_X)
B_X = np.zeros((p_X, 1)); B_X[-1, 0] = params_price['sigma']
c_X = np.zeros(p_X); c_X[:len(b_X)] = b_X

a_Y = np.array(params_temp['a_coeffs'])
b_Y = np.array(params_temp['b_coeffs'])
p_Y = len(a_Y)
A_Y = build_companion(a_Y)
B_Y = np.zeros((p_Y, 1)); B_Y[-1, 0] = params_temp['sigma']
c_Y = np.zeros(p_Y); c_Y[:len(b_Y)] = b_Y

# Coupling matrix Gamma = gamma0 * B_X
gamma0 = float(coupling['gamma0'])
Gamma  = gamma0 * B_X  # (p_X, 1)

print('Matrices built:')
print(f'  A_X shape: {A_X.shape},  A_Y shape: {A_Y.shape}')
print(f'  gamma0    = {gamma0:.6f}')
print(f'  NIG_X: alpha={nig_X["alpha"]:.4f}  beta={nig_X["beta"]:.4f}  '
      f'mu={nig_X["mu"]:.6f}  delta={nig_X["delta"]:.6f}')
print(f'  NIG_Y: alpha={nig_Y["alpha"]:.4f}  beta={nig_Y["beta"]:.4f}  '
      f'mu={nig_Y["mu"]:.6f}  delta={nig_Y["delta"]:.6f}')

## 1.  Contract parameters and current state

In [ ]:
# Use stationary mean state (zero) as starting point
Z_X0 = np.zeros(p_X)
Z_Y0 = np.zeros(p_Y)

# Seasonal offsets (set to zero for the test; Lambda_S and Lambda_Y can be
# loaded from the seasonal fit if available)
Lambda_X = 0.0
Lambda_Y = 0.0

# Compute forward price at tau=1 month
tau_1m  = 1.0 / 12.0   # 1 month in years
tau_1w  = 1.0 / 52.0   # 1 week
tau_3m  = 3.0 / 12.0
tau_6m  = 6.0 / 12.0

F_1m = compute_forward_price(
    tau_1m, Z_X0, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma,
    nig_X, nig_Y, Lambda_X=Lambda_X
)
F_1w = compute_forward_price(
    tau_1w, Z_X0, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma,
    nig_X, nig_Y, Lambda_X=Lambda_X
)

print(f'Forward prices at stationary state:')
print(f'  F(t, t+1w)  = {F_1w:.4f}')
print(f'  F(t, t+1m)  = {F_1m:.4f}')

# Strikes: ATM and OTM
K_S_atm = F_1m   # at-the-money
K_Y_atm = float(c_Y @ Z_Y0)   # at current Y level
print(f'\nStrikes: K_S (ATM) = {K_S_atm:.4f},  K_Y (ATM) = {K_Y_atm:.4f}')

## 2.  Monte Carlo simulation

Simulate $N=100{,}000$ paths of the coupled CARMA system over each maturity
using the exact matrix-exponential propagator with NIG increments.

In [ ]:
N_PATHS = 100_000
rng = np.random.default_rng(42)

mc_results = {}

for label, tau in [('1wk', tau_1w), ('1mo', tau_1m), ('3mo', tau_3m)]:
    # Number of daily steps
    n_steps = max(int(tau * 365), 1)
    dt_sim  = tau / n_steps

    print(f'Simulating {label} ({n_steps} steps × {N_PATHS:,} paths) …', end=' ')

    X_paths, Y_paths, _, _ = simulate_coupled_carma(
        A_X, B_X, c_X, A_Y, B_Y, c_Y, Gamma,
        n_steps=n_steps, dt=dt_sim,
        nig_X=nig_X, nig_Y=nig_Y,
        n_paths=N_PATHS, rng=rng,
    )

    X_T = X_paths[:, -1]   # terminal log-price factor
    Y_T = Y_paths[:, -1]   # terminal temperature factor
    S_T = np.exp(Lambda_X + X_T)   # spot price (seasonal=0)

    # Forward price (re-compute at this tau)
    F_tau = compute_forward_price(
        tau, Z_X0, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma,
        nig_X, nig_Y, Lambda_X=Lambda_X
    )
    K_S = F_tau                         # ATM
    K_Y = float(c_Y @ Z_Y0)             # current Y level

    # Payoffs
    call_payoff  = np.maximum(S_T - K_S, 0.0)
    ind_quanto   = (S_T > K_S).astype(float) * (Y_T > K_Y).astype(float)
    lin_quanto   = np.maximum(S_T - K_S, 0.0) * (Y_T - K_Y)

    mc_results[label] = {
        'tau': tau,
        'K_S': K_S, 'K_Y': K_Y,
        'call_mc':    float(call_payoff.mean()),
        'call_se':    float(call_payoff.std() / np.sqrt(N_PATHS)),
        'ind_mc':     float(ind_quanto.mean()),
        'ind_se':     float(ind_quanto.std()  / np.sqrt(N_PATHS)),
        'lin_mc':     float(lin_quanto.mean()),
        'lin_se':     float(lin_quanto.std()  / np.sqrt(N_PATHS)),
    }
    print(f'done  call={mc_results[label]["call_mc"]:.4f}±{mc_results[label]["call_se"]:.4f}')

print('\nMC complete.')

## 3.  Fourier pricing

In [ ]:
fourier_results = {}

for label, tau in [('1wk', tau_1w), ('1mo', tau_1m), ('3mo', tau_3m)]:
    res = mc_results[label]
    K_S, K_Y = res['K_S'], res['K_Y']

    print(f'Fourier pricing {label} …', end=' ')

    # Plain call via Carr-Madan FFT
    call_fft = fourier_price_1d(
        tau, Z_X0, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma,
        nig_X, nig_Y, K=K_S, Lambda_X=Lambda_X,
    )

    # Indicator quanto via 2D Fourier
    ind_fft = fourier_price_indicator_quanto(
        tau, Z_X0, Z_Y0, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma,
        nig_X, nig_Y, K_S=K_S, K_Y=K_Y,
        Lambda_X=Lambda_X, Lambda_Y=Lambda_Y,
    )

    fourier_results[label] = {
        'call_fft': call_fft,
        'ind_fft':  ind_fft,
    }
    print(f'call={call_fft:.4f}  ind={ind_fft:.4f}')

print('\nFourier complete.')

## 4.  Comparison table

In [ ]:
rows = []
for label in ['1wk', '1mo', '3mo']:
    mc  = mc_results[label]
    fft = fourier_results[label]

    for payoff, fft_val, mc_val, mc_se in [
        ('Plain call',      fft['call_fft'], mc['call_mc'], mc['call_se']),
        ('Indicator quanto', fft['ind_fft'],  mc['ind_mc'],  mc['ind_se']),
    ]:
        rel_err = abs(fft_val - mc_val) / (mc_val + 1e-12)
        within  = abs(fft_val - mc_val) < 2 * mc_se
        rows.append({
            'Payoff': payoff, 'Maturity': label,
            'V_FFT': fft_val,
            'V_MC':  mc_val,
            'MC_SE': mc_se,
            'Rel_err': rel_err,
            'Within_2sigma': within,
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False, float_format=lambda x: f'{x:.5f}'))
df.to_csv(RES_DIR / 'pricing_comparison.csv', index=False)
print(f'\nSaved → {RES_DIR}/pricing_comparison.csv')
print(f"Max relative error: {df['Rel_err'].max():.4f}")
print(f"All within 2 sigma: {df['Within_2sigma'].all()}")

## 5.  Diagnostic plot: price vs strike curve

In [ ]:
# Price vs strike for indicator quanto at 1-month maturity
tau = tau_1m
n_steps_sim = max(int(tau * 365), 1)
dt_sim = tau / n_steps_sim

# Simulate once for all strikes
rng2 = np.random.default_rng(123)
X_paths, Y_paths, _, _ = simulate_coupled_carma(
    A_X, B_X, c_X, A_Y, B_Y, c_Y, Gamma,
    n_steps=n_steps_sim, dt=dt_sim,
    nig_X=nig_X, nig_Y=nig_Y,
    n_paths=50_000, rng=rng2,
)
X_T = X_paths[:, -1]; Y_T = Y_paths[:, -1]
S_T = np.exp(Lambda_X + X_T)
F_1m_plot = compute_forward_price(
    tau, Z_X0, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma,
    nig_X, nig_Y, Lambda_X=Lambda_X
)
Y0_curr = float(c_Y @ Z_Y0)

# Strike grids
K_S_grid = F_1m_plot * np.linspace(0.7, 1.3, 25)
K_Y_fixed = Y0_curr  # fix secondary at ATM

ind_mc_grid  = [np.mean((S_T > k) * (Y_T > K_Y_fixed)) for k in K_S_grid]
ind_fft_grid = [fourier_price_indicator_quanto(
    tau, Z_X0, Z_Y0, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma,
    nig_X, nig_Y, K_S=float(k), K_Y=K_Y_fixed,
) for k in K_S_grid]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(K_S_grid / F_1m_plot, ind_mc_grid,  'o', ms=4, alpha=0.8, label='MC (50k paths)')
ax.plot(K_S_grid / F_1m_plot, ind_fft_grid, '-', lw=2, label='Fourier')
ax.set_xlabel('Strike $K_S / F(t,T)$')
ax.set_ylabel('Indicator quanto price')
ax.set_title('Indicator quanto $\\mathbf{1}_{S_T>K_S}\\mathbf{1}_{Y_T>K_Y}$ — 1 month, ATM secondary')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'pricing_mc_comparison.png', dpi=150)
plt.show()
print('Saved pricing_mc_comparison.png')

## 6.  Quanto adjustment: coupling vs independence

In [ ]:
# Set Gamma=0 (independent model) and compare to coupled model
Gamma_zero = np.zeros_like(Gamma)

adj_rows = []
for label, tau in [('1wk', tau_1w), ('1mo', tau_1m), ('3mo', tau_3m), ('6mo', tau_6m)]:
    K_S = compute_forward_price(tau, Z_X0, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma,
                                nig_X, nig_Y, Lambda_X=Lambda_X)
    K_Y = float(c_Y @ Z_Y0)

    V_coupled = fourier_price_indicator_quanto(
        tau, Z_X0, Z_Y0, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma,
        nig_X, nig_Y, K_S=K_S, K_Y=K_Y, Lambda_X=Lambda_X, Lambda_Y=Lambda_Y,
    )
    V_indep = fourier_price_indicator_quanto(
        tau, Z_X0, Z_Y0, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma_zero,
        nig_X, nig_Y, K_S=K_S, K_Y=K_Y, Lambda_X=Lambda_X, Lambda_Y=Lambda_Y,
    )
    adj = V_coupled - V_indep
    adj_pct = 100 * adj / (V_indep + 1e-12)
    adj_rows.append({'Maturity': label, 'tau': tau, 'V_coupled': V_coupled,
                     'V_indep': V_indep, 'Delta_V': adj, 'Delta_V_pct': adj_pct})
    print(f'{label}: V(coupled)={V_coupled:.4f}  V(indep)={V_indep:.4f}  '
          f'ΔV={adj:+.4f}  ({adj_pct:+.2f}%)')

df_adj = pd.DataFrame(adj_rows)
df_adj.to_csv(RES_DIR / 'quanto_adjustment.csv', index=False)
print(f'Saved → {RES_DIR}/quanto_adjustment.csv')